# Automatic Crime Detection — Final Thesis Training (Google Colab GPU)

Notebook-kan waa **final thesis mode**, mana aha quick benchmark.

- 70% train / 15% validation / 15% untouched hold-out test
- TF-IDF waxaa lagu fit-gareeyaa train only
- Dhammaan 7 classical models: full randomized hyperparameter tuning
- 1D-CNN iyo BiLSTM: full-data architecture tuning + early stopping
- SomBERTa-A: full data, max length 128, 3 epochs
- SomBERTa-B: full data, max length 256, 4 epochs
- Final comparison: Accuracy, Precision, Recall, F1, Crime Recall, iyo runtime

**Colab:** `Runtime → Change runtime type → T4 GPU`, kadib `Runtime → Run all`.


In [ ]:
# Preserve Colab's CUDA-enabled torch; install tested NLP dependencies.
%pip install -q -U transformers==5.14.1 sentencepiece accelerate safetensors
print('NLP dependencies installed; Colab CUDA torch was preserved.')

In [ ]:
import re
import sys
from functools import lru_cache
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import joblib

_SOMALI_STOPWORDS = frozenset(['aad', 'aadka', 'aan', 'aanad', 'aaney', 'aay', 'aaynu', 'adiga', 'ah', 'aha', 'ahaa', 'ahaanba', 'ahaatee', 'ahayd', 'ahayn', 'aheyn', 'ahna', 'ama', 'amase', 'amma', 'anaan', 'aniga', 'awgeed', 'awgii', 'ay', 'aya', 'ayaa', 'ayaad', 'ayaana', 'ayay', 'ayayna', 'ayee', 'ayey', 'ayna', 'aynu', 'aysan', 'ayuu', 'baa', 'baad', 'baan', 'baannu', 'bal', 'beey', 'dabadeed', 'dadka', 'dhan', 'dib', 'ee', 'eey', 'eeyga', 'gabi', 'guud', 'ha', 'haa', 'haatan', 'hadana', 'hadda', 'haddaba', 'haddana', 'haddii', 'hadduu', 'hadii', 'haku', 'halkaasoo', 'hase', 'hasii', 'hasoo', 'hor', 'hore', 'horeba', 'horta', 'idiin', 'idiinku', 'idin', 'idinka', 'iga', 'igu', 'ii', 'iigu', 'iima', 'iimaanu', 'ila', 'ilaa', 'in', 'ina', 'inaa', 'inaad', 'inaan', 'inaana', 'inaanay', 'inaaney', 'inaanu', 'inaga', 'inan', 'inay', 'inaysan', 'iney', 'inkasta', 'inkastoo', 'innaga', 'innoo', 'inoo', 'inta', 'intaa', 'intaas', 'intaysan', 'intii', 'intiisa', 'inuu', 'inuusan', 'is', 'isaga', 'isagoo', 'iska', 'iskaba', 'iskeed', 'isku', 'iskugu', 'isla', 'islamarkaana', 'isoo', 'isu', 'isugu', 'iwm', 'iyada', 'iyadoo', 'iyaga', 'iyo', 'jeer', 'jirta', 'ka', 'kaa', 'kaas', 'kaasi', 'kaasoo', 'kaddib', 'kaddibna', 'kadib', 'kadibna', 'kaga', 'kahor', 'kala', 'kalana', 'kale', 'kaleba', 'kalena', 'kaliya', 'kama', 'kamid', 'kamida', 'kan', 'kana', 'kani', 'kasii', 'kasoo', 'kasta', 'kastaba', 'kii', 'ku', 'kugu', 'kula', 'kulasoo', 'kule', 'kuma', 'kuna', 'kusii', 'kusoo', 'kuu', 'kuugu', 'kuwa', 'kuwaas', 'kuwaasi', 'kuwaasoo', 'kuwee', 'kuwii', 'kuwo', 'la', 'laakiin', 'laakin', 'laga', 'lagala', 'lagama', 'lagu', 'laguma', 'laguna', 'lakiin', 'lakin', 'lala', 'lama', 'lamana', 'lasoo', 'lee', 'leh', 'lkn', 'loo', 'looga', 'loogu', 'looguna', 'loona', 'mana', 'marka', 'markaa', 'markaana', 'markii', 'markiiba', 'markuu', 'marna', 'maxaa', 'maxaad', 'maxaan', 'maxad', 'maxay', 'mid', 'midba', 'midna', 'mise', 'miyuu', 'mooyee', 'muxuu', 'naga', 'nagala', 'nagu', 'naloo', 'noo', 'nooga', 'nugul', 'oo', 'qof', 'sheegay', 'si', 'sida', 'sidaas', 'sidee', 'sideen', 'sidii', 'sidoo', 'sii', 'soo', 'ta', 'taan', 'taas', 'taasi', 'tani', 'tankale', 'u', 'uga', 'ugu', 'ula', 'uma', 'una', 'unbuu', 'usoo', 'uu', 'uugu', 'uuna', 'uusan', 'waa', 'waad', 'waana', 'waase', 'waayahay', 'waaye', 'waayo', 'wada', 'walba', 'walbo', 'wali', 'waliba', 'walina', 'wax', 'waxa', 'waxaa', 'waxaad', 'waxaan', 'waxaana', 'waxaanay', 'waxaanu', 'waxaas', 'waxan', 'waxana', 'waxay', 'waxayna', 'waxba', 'waxey', 'waxsoo', 'waxuu', 'way', 'weeyaan', 'weligeed', 'weligii', 'wixii', 'wuu', 'wuxi', 'wuxu', 'wuxuu', 'wuxuuna', 'xataa', 'xitaa', 'yaa', 'yaan', 'yahay', 'yeeshee', 'yihiin'])

@lru_cache(maxsize=1)
def load_somali_stopwords():
    return _SOMALI_STOPWORDS

def clean_text(text):
    if text is None:
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+|@\w+|#\w+|\d+', ' ', text)
    text = re.sub(r"[^a-z'\s]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_text(text, remove_stopwords=True):
    tokens = clean_text(text).split()
    if remove_stopwords:
        tokens = [t for t in tokens if t.strip("'") not in _SOMALI_STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

FAST_RUN = False
RANDOM_STATE = 42
CRIME_LABEL = 'crime-related'
NON_CRIME_LABEL = 'not crime-related'
OUTPUT_DIR = Path('/content/final_thesis_artifacts/figures')
AI_MODEL_DIR = Path('/content/final_thesis_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AI_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('FINAL THESIS MODE:', not FAST_RUN)
print('Somali stopwords:', len(load_somali_stopwords()))
print('Artifacts directory:', AI_MODEL_DIR)


## 1. Data Loading & Cleaning

Cleaning-ku wuxuu ka mid yahay: label normalize, missing, **duplicates** (sida model sax), short-text filter.


In [ ]:
# Locate the dataset in Colab; ask for upload only when it is absent.
dataset_candidates = [
    Path('/content/dataset.csv.csv'),
    Path('/content/dataset_cleaned.csv'),
    Path('dataset.csv.csv'),
    Path('dataset_cleaned.csv'),
]
dataset_path = next((p for p in dataset_candidates if p.is_file()), None)

if dataset_path is None:
    try:
        from google.colab import files
        print('Upload dataset.csv.csv (columns required: text, category)')
        uploaded = files.upload()
        dataset_path = Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError(
            'Dataset not found. Upload dataset.csv.csv to the Colab runtime.'
        ) from exc

try:
    df = pd.read_csv(dataset_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(dataset_path, encoding='latin1')

df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df = df.dropna(subset=['text', 'category']).copy()
df['text'] = df['text'].astype(str)
df['category'] = df['category'].astype(str).str.strip().str.lower()
df['category'] = df['category'].replace({
    'crime': CRIME_LABEL,
    'crime related': CRIME_LABEL,
    'not crime': NON_CRIME_LABEL,
    'not crime related': NON_CRIME_LABEL,
})
df = df[df['category'].isin([CRIME_LABEL, NON_CRIME_LABEL])].copy()
df['text'] = df['text'].str.strip()

print('Dataset:', dataset_path.resolve())
print('Rows loaded:', len(df))
print('Category counts:\n', df['category'].value_counts())


In [ ]:
# --- From model sax: Duplicate analysis ---
n_before = len(df)
n_dupes = int(df.duplicated(subset=['text']).sum())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Duplicates', 'Unique texts'], [n_dupes, n_before - n_dupes],
       color=['#e74c3c', '#27ae60'], edgecolor='black')
ax.set_title('Duplicate Analysis (from model sax)')
ax.set_ylabel('Count')
for i, v in enumerate([n_dupes, n_before - n_dupes]):
    ax.text(i, v + max(n_before * 0.01, 5), str(v), ha='center', fontweight='bold')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '00_duplicates.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df = df[df['text'].str.len() >= 40].copy()

print(f'Removed {n_dupes} duplicate texts | Remaining: {len(df)}')

In [ ]:
# Shared preprocessing (sax stopwords + sax cleaning rules inside preprocessing.py)
df['cleaned_text'] = df['text'].apply(clean_text)
df['preprocessed_text'] = df['text'].apply(preprocess_text)
df = df[df['preprocessed_text'].str.len() >= 20].reset_index(drop=True)

df['text_length'] = df['text'].str.len()
df['word_count'] = df['preprocessed_text'].str.split().str.len()
df['sentence_length'] = df['cleaned_text'].str.len()  # sax naming

print(f'Documents after full cleaning: {len(df)}')
print(df['category'].value_counts())
print('\n--- BEFORE vs AFTER (sample 0, like model sax) ---')
print('BEFORE:', df['text'].iloc[0][:220], '...')
print('AFTER :', df['preprocessed_text'].iloc[0][:220], '...')


## 2. Visualizations & EDA

Waxyaabaha muhiimka ee laga soo qaatay **model sax** + charts pipeline-ka.


In [ ]:
# Category balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['category'].value_counts()
colors = ['#c0392b', '#2980b9']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Qeybinta Categories (tirada documents)')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Tirada (count)')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.02, 0.02))
axes[1].set_title('Saamiga Categories (%)')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '01_category_balance.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

In [ ]:
# Text length + crime vs not-crime averages (model sax comparison)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cat, color in zip([CRIME_LABEL, NON_CRIME_LABEL], ['#c0392b', '#2980b9']):
    subset = df.loc[df['category'] == cat, 'word_count']
    axes[0].hist(subset, bins=35, alpha=0.55, label=cat, color=color, edgecolor='black')
axes[0].set_title('Word Count Distribution by Category')
axes[0].set_xlabel('Words (after preprocess)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

crime_df = df[df['category'] == CRIME_LABEL]
notcrime_df = df[df['category'] == NON_CRIME_LABEL]
comparison = pd.DataFrame({
    'Crime': [crime_df['sentence_length'].mean(), crime_df['word_count'].mean()],
    'Not Crime': [notcrime_df['sentence_length'].mean(), notcrime_df['word_count'].mean()],
}, index=['Sentence Length', 'Word Count'])
comparison.plot(kind='bar', ax=axes[1], color=['#c0392b', '#2980b9'], edgecolor='black')
axes[1].set_title('Crime vs Not Crime Averages (from model sax)')
axes[1].set_ylabel('Average')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '02_text_length_comparison.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()
print(comparison.round(1))

In [ ]:
# Top words per class (sax + side-by-side)
def top_words(texts, n=15):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    return counter.most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, cat, color in zip(axes, [CRIME_LABEL, NON_CRIME_LABEL], ['#c0392b', '#2980b9']):
    words = top_words(df.loc[df['category'] == cat, 'preprocessed_text'], 15)
    labels = [w for w, _ in words][::-1]
    vals = [c for _, c in words][::-1]
    ax.barh(labels, vals, color=color, edgecolor='black')
    ax.set_title(f'Top 15 words — {cat}')
    ax.set_xlabel('Frequency')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '03_top_words_by_class.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

print('Top crime words:', top_words(crime_df['preprocessed_text'], 10))
print('Top not-crime words:', top_words(notcrime_df['preprocessed_text'], 10))

In [ ]:
# Overall most common words + word length distribution (from model sax)
all_words = ' '.join(df['preprocessed_text']).split()
word_freq = Counter(all_words)
common_words = word_freq.most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
words = [w for w, _ in common_words]
counts = [c for _, c in common_words]
axes[0].bar(words, counts, color='#34495e', edgecolor='black')
axes[0].set_title('Most Common Words (all data)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Frequency')
axes[0].tick_params(axis='x', rotation=55)

word_lengths = [len(w) for w in all_words]
axes[1].hist(word_lengths, bins=20, color='#16a085', edgecolor='black')
axes[1].set_title('Word Length Distribution (from model sax)')
axes[1].set_xlabel('Word length (chars)')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '03b_word_freq_and_length.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

In [ ]:
# Word clouds — general / crime / not-crime (from model sax)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clouds = [
    ('All texts', ' '.join(df['preprocessed_text']), 'viridis'),
    ('Crime-related', ' '.join(crime_df['preprocessed_text']), 'Reds'),
    ('Not crime-related', ' '.join(notcrime_df['preprocessed_text']), 'Blues'),
]
for ax, (title, text, cmap) in zip(axes, clouds):
    if text.strip():
        wc = WordCloud(width=900, height=500, background_color='white',
                       colormap=cmap, max_words=120).generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '03c_wordclouds.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()
print('Saved wordclouds (model sax style)')

## 3. Train / Validation / Test Split (NO leakage)

Split **ka hor** TF-IDF — ka duwan model sax oo fit_transform sameeyay data-da oo dhan.


In [ ]:
X_text = df['preprocessed_text']
y = df['category']

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(X_train_text), y_train.value_counts().to_dict())
print('Val  :', len(X_val_text), y_val.value_counts().to_dict())
print('Test :', len(X_test_text), y_test.value_counts().to_dict())


## 4. Feature Engineering (fit on TRAIN only)


In [ ]:
def make_vectorizer(max_features=10000, ngram_range=(1, 2), analyzer='word'):
    return TfidfVectorizer(
        max_features=max_features,
        min_df=2,
        max_df=0.92,
        ngram_range=ngram_range,
        analyzer=analyzer,
        sublinear_tf=True,
    )

vectorizer = make_vectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

print('TF-IDF features:', X_train.shape[1])
print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)


## 5. Model Training (baseline)

Models-ka sida sax + Gradient Boosting / LinearSVC. Metrics waxaa ku jira **Crime Recall**.


In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Naive Bayes': MultinomialNB(alpha=0.5),
    'Decision Tree': DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=25, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, random_state=RANDOM_STATE),
    'Linear SVM': LinearSVC(max_iter=3000, class_weight='balanced', dual=False, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=7),
}

def evaluate_model(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        'accuracy': accuracy_score(y_eval, pred),
        'precision': precision_score(y_eval, pred, average='weighted', zero_division=0),
        'recall': recall_score(y_eval, pred, average='weighted', zero_division=0),
        'f1': f1_score(y_eval, pred, average='weighted', zero_division=0),
        'crime_recall': recall_score(y_eval, pred, pos_label=CRIME_LABEL, zero_division=0),
        'predictions': pred,
    }

trained = {}
baseline_rows = []

print('Training baseline models...\n')
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    metrics = evaluate_model(model, X_val, y_val)
    baseline_rows.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'Crime Recall': metrics['crime_recall'],
    })
    print(f"  {name:22s}  F1={metrics['f1']:.4f}  CrimeRecall={metrics['crime_recall']:.4f}")
    # Per-model report like model sax (validation)
    print(classification_report(y_val, metrics['predictions'], digits=3))

baseline_df = pd.DataFrame(baseline_rows).sort_values('F1', ascending=False).reset_index(drop=True)
print('\\n=== BASELINE VALIDATION ===')
baseline_df


In [ ]:
plot_df = baseline_df.sort_values('F1')
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = np.arange(len(plot_df))
ax.barh(y_pos, plot_df['F1'], color='#27ae60', edgecolor='black', height=0.45, label='F1')
ax.barh(y_pos + 0.45, plot_df['Crime Recall'], color='#e67e22', edgecolor='black', height=0.45, label='Crime Recall')
ax.set_yticks(y_pos + 0.225)
ax.set_yticklabels(plot_df['Model'])
ax.set_xlim(0, 1.05)
ax.set_xlabel('Score (0–1)')
ax.set_title('Baseline Models — Validation F1 vs Crime Recall')
ax.legend(loc='lower right')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '04_baseline_comparison.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()
print('Best baseline:', baseline_df.iloc[0]['Model'])

## 6. Optimization — Feature Configuration Search


In [ ]:
feature_configs = [
    {'name': 'word_3k_(1,1)', 'max_features': 3000, 'ngram_range': (1, 1), 'analyzer': 'word'},
    {'name': 'word_5k_(1,1)', 'max_features': 5000, 'ngram_range': (1, 1), 'analyzer': 'word'},  # model sax default size
    {'name': 'word_10k_(1,2)', 'max_features': 10000, 'ngram_range': (1, 2), 'analyzer': 'word'},
    {'name': 'word_15k_(1,3)', 'max_features': 15000, 'ngram_range': (1, 3), 'analyzer': 'word'},
    {'name': 'char_8k_(3,5)', 'max_features': 8000, 'ngram_range': (3, 5), 'analyzer': 'char_wb'},
]

opt_rows = []
print('Feature optimization...\n')
for cfg in feature_configs:
    vec = make_vectorizer(cfg['max_features'], cfg['ngram_range'], cfg['analyzer'])
    Xt = vec.fit_transform(X_train_text)
    Xv = vec.transform(X_val_text)
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    clf.fit(Xt, y_train)
    m = evaluate_model(clf, Xv, y_val)
    opt_rows.append({'Config': cfg['name'], 'Features': Xt.shape[1], 'F1': m['f1'],
                     'Crime Recall': m['crime_recall'], 'Accuracy': m['accuracy']})
    print(f"  {cfg['name']:18s} feats={Xt.shape[1]:5d}  F1={m['f1']:.4f}")

opt_df = pd.DataFrame(opt_rows).sort_values('F1', ascending=False).reset_index(drop=True)
best_feature_name = opt_df.iloc[0]['Config']
print('\\nBest feature config:', best_feature_name)
opt_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(opt_df))
width = 0.35
ax.bar(x - width/2, opt_df['F1'], width, label='F1', color='#27ae60', edgecolor='black')
ax.bar(x + width/2, opt_df['Crime Recall'], width, label='Crime Recall', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(opt_df['Config'], rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Optimization: TF-IDF Feature Configs (includes sax 5k)')
ax.legend()
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '05_feature_optimization.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

best_cfg = next(c for c in feature_configs if c['name'] == best_feature_name)
vectorizer = make_vectorizer(best_cfg['max_features'], best_cfg['ngram_range'], best_cfg['analyzer'])
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)
print('Using:', best_feature_name, X_train.shape)

## 7. Hyperparameter Tuning (ma jirin model sax — waa cusub)


In [ ]:
from scipy.sparse import vstack

cv = StratifiedKFold(n_splits=2 if FAST_RUN else 3, shuffle=True, random_state=RANDOM_STATE)
X_tune = vstack([X_train, X_val])
y_tune = pd.concat([y_train, y_val], ignore_index=True)

tuning_spaces = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2500, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': np.logspace(-2, 2, 10), 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'saga']},
    ),
    'Naive Bayes': (
        MultinomialNB(),
        {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0], 'fit_prior': [True, False]},
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        {'max_depth': [10, 20, 30, None], 'min_samples_split': [2, 5, 10],
         'min_samples_leaf': [1, 2, 4], 'criterion': ['gini', 'entropy']},
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [120, 200, 300], 'max_depth': [15, 25, 40, None],
         'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
         'max_features': ['sqrt', 0.3]},
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'n_estimators': [80, 120, 180], 'learning_rate': [0.03, 0.05, 0.1],
         'max_depth': [2, 3, 4], 'subsample': [0.8, 1.0]},
    ),
    'Linear SVM': (
        LinearSVC(max_iter=4000, class_weight='balanced', dual=False, random_state=RANDOM_STATE),
        {'C': np.logspace(-2, 2, 12)},
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': [3, 5, 7, 9, 11], 'weights': ['uniform', 'distance'],
         'metric': ['cosine', 'euclidean']},
    ),
}

tuned_models = {}
tune_rows = []
print('Hyperparameter tuning...\n')
for name, (estimator, params) in tuning_spaces.items():
    search = RandomizedSearchCV(
        estimator, param_distributions=params, scoring='f1_weighted',
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True,
        n_iter=3 if FAST_RUN else 12,
    )
    search.fit(X_tune, y_tune)
    tuned_models[name] = search.best_estimator_
    tune_rows.append({'Model': name, 'Best CV F1': search.best_score_, 'Best Params': str(search.best_params_)})
    print(f"  {name:22s}  CV F1={search.best_score_:.4f}")
    print(f"    params: {search.best_params_}")

tune_df = pd.DataFrame(tune_rows).sort_values('Best CV F1', ascending=False).reset_index(drop=True)
tune_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_t = tune_df.sort_values('Best CV F1')
ax.barh(plot_t['Model'], plot_t['Best CV F1'], color='#8e44ad', edgecolor='black')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Cross-validated F1 (weighted)')
ax.set_title('Hyperparameter Tuning Results (3-fold CV)')
for i, v in enumerate(plot_t['Best CV F1']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '06_hyperparameter_tuning.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

## 8. Model Testing (hold-out TEST)


In [ ]:
test_rows = []
test_preds = {}
for name, model in tuned_models.items():
    m = evaluate_model(model, X_test, y_test)
    test_preds[name] = m['predictions']
    test_rows.append({
        'Model': name,
        'Accuracy': m['accuracy'],
        'F1': m['f1'],
        'Crime Precision': precision_score(y_test, m['predictions'], pos_label=CRIME_LABEL, zero_division=0),
        'Crime Recall': m['crime_recall'],
    })

best_baseline_name = baseline_df.iloc[0]['Model']
from sklearn.base import clone
baseline_for_test = clone(trained[best_baseline_name])
baseline_for_test.fit(X_train, y_train)
m_base = evaluate_model(baseline_for_test, X_test, y_test)
test_rows.append({
    'Model': f'{best_baseline_name} (baseline)',
    'Accuracy': m_base['accuracy'],
    'F1': m_base['f1'],
    'Crime Precision': precision_score(y_test, m_base['predictions'], pos_label=CRIME_LABEL, zero_division=0),
    'Crime Recall': m_base['crime_recall'],
})

test_df = pd.DataFrame(test_rows).sort_values('F1', ascending=False).reset_index(drop=True)
print('=== HOLD-OUT TEST ===')
print(test_df.to_string(index=False))

production_name = max(tuned_models.keys(), key=lambda n: f1_score(y_test, test_preds[n], average='weighted'))
production_model = tuned_models[production_name]
print('\\nSelected production model:', production_name)


In [ ]:
y_pred = production_model.predict(X_test)
labels_order = [NON_CRIME_LABEL, CRIME_LABEL]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: NOT crime', 'Pred: CRIME'],
            yticklabels=['True: NOT crime', 'True: CRIME'])
axes[0].set_title(f'Confusion Matrix — {production_name}')

y_bin = (y_test == CRIME_LABEL).astype(int)
if hasattr(production_model, 'predict_proba'):
    scores = production_model.predict_proba(X_test)[:, list(production_model.classes_).index(CRIME_LABEL)]
elif hasattr(production_model, 'decision_function'):
    scores = production_model.decision_function(X_test)
else:
    scores = (y_pred == CRIME_LABEL).astype(float)

fpr, tpr, _ = roc_curve(y_bin, scores)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#c0392b', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (crime-related)')
axes[1].legend(loc='lower right')
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '07_test_confusion_roc.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_t = test_df.copy()
x = np.arange(len(plot_t))
w = 0.2
ax.bar(x - 1.5*w, plot_t['Accuracy'], w, label='Accuracy', color='#3498db', edgecolor='black')
ax.bar(x - 0.5*w, plot_t['F1'], w, label='F1', color='#27ae60', edgecolor='black')
ax.bar(x + 0.5*w, plot_t['Crime Precision'], w, label='Crime Precision', color='#9b59b6', edgecolor='black')
ax.bar(x + 1.5*w, plot_t['Crime Recall'], w, label='Crime Recall', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(plot_t['Model'], rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Testing — Hold-out Test Metrics')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
try:
    try:
        plt.savefig(OUTPUT_DIR / '08_model_testing.png', dpi=200, bbox_inches='tight')
    except OSError as chart_error:
        print(f'Chart save skipped: {chart_error}')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

In [ ]:
fn_mask = (y_test == CRIME_LABEL) & (y_pred != CRIME_LABEL)
fp_mask = (y_test == NON_CRIME_LABEL) & (y_pred == CRIME_LABEL)
print(f'False Negatives (missed crime): {fn_mask.sum()}')
print(f'False Positives (false alarm): {fp_mask.sum()}')
print('\\n--- Sample missed crimes ---')
for i, t in enumerate(X_test_text[fn_mask].head(5), 1):
    print(f'{i}. {t[:180]}...')
print('\\n--- Sample false alarms ---')
for i, t in enumerate(X_test_text[fp_mask].head(5), 1):
    print(f'{i}. {t[:180]}...')


## 9. Deep Learning & Transformer Models

Qaybtani waxay isticmaashaa **isla train/validation/test split-ka** kore si natiijooyinka loo barbar dhigi karo.

- **1D-CNN:** local crime phrases / n-gram patterns
- **BiLSTM:** xiriirka ereyada labada jiho
- **SomBERTa-A:** max_length=128, learning rate=2e-5
- **SomBERTa-B:** max_length=256, learning rate=1e-5

Default-ku waa **FULL training**: model kasta wuxuu helayaa isla train/validation/test rows. Hal mar `Run All` ayaa wada tababaraya dhammaan models-ka.

In [ ]:
# Final SomBERTa training should use a Colab GPU.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'GPU lama helin. Dooro Runtime > Change runtime type > T4 GPU, kadib Run all.'
    )

In [ ]:
# Reproducible deep-learning configuration
import copy
import gc
import os
import random
import time
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

FAST_RUN = False
SEED = RANDOM_STATE
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'Execution mode: {"FAST CPU benchmark" if FAST_RUN else "FULL training — same split"}')
assert not FAST_RUN, 'Final comparison requires FAST_RUN=False'
print('All models use the same train/validation/test row indices.')

### 9.1 Neural text preparation

Vocabulary-ga waxaa laga dhisayaa **train only** si looga hortago data leakage. `<UNK>` wuxuu qabtaa ereyada aan train-ka ku jirin, halka `<PAD>` uu simayo dhererka sequences-ka.

In [ ]:
# Use the already-cleaned text and the exact same split indices
label_to_id = {NON_CRIME_LABEL: 0, CRIME_LABEL: 1}

train_texts_dl = X_train_text.astype(str).tolist()
val_texts_dl = X_val_text.astype(str).tolist()
test_texts_dl = X_test_text.astype(str).tolist()
y_train_dl = y_train.map(label_to_id).astype(int).tolist()
y_val_dl = y_val.map(label_to_id).astype(int).tolist()
y_test_dl = y_test.map(label_to_id).astype(int).tolist()

# Preserve full labels for the independent transformer sampling below.
y_train_all_dl, y_val_all_dl, y_test_all_dl = y_train_dl.copy(), y_val_dl.copy(), y_test_dl.copy()

def quick_stratified_pair(texts, labels, limit):
    if not FAST_RUN or len(labels) <= limit:
        return list(texts), list(labels)
    chosen, _ = train_test_split(
        np.arange(len(labels)), train_size=limit, random_state=SEED, stratify=labels
    )
    return [texts[i] for i in chosen], [int(labels[i]) for i in chosen]

train_texts_dl, y_train_dl = quick_stratified_pair(train_texts_dl, y_train_dl, 1000)
val_texts_dl, y_val_dl = quick_stratified_pair(val_texts_dl, y_val_dl, 250)
test_texts_dl, y_test_dl = quick_stratified_pair(test_texts_dl, y_test_dl, 300)

token_counts = Counter(token for text in train_texts_dl for token in text.split())
vocab = {'<PAD>': 0, '<UNK>': 1}
for token, count in token_counts.most_common(30000):
    if count >= 2:
        vocab[token] = len(vocab)

MAX_LEN_DL = 160

def encode_words(text):
    ids = [vocab.get(token, 1) for token in str(text).split()[:MAX_LEN_DL]]
    return torch.tensor(ids or [1], dtype=torch.long)

class WordDataset(Dataset):
    def __init__(self, texts, labels):
        self.items = [(encode_words(t), int(y)) for t, y in zip(texts, labels)]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

def collate_words(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

train_word_loader = DataLoader(WordDataset(train_texts_dl, y_train_dl), batch_size=64, shuffle=True,
                               collate_fn=collate_words)
val_word_loader = DataLoader(WordDataset(val_texts_dl, y_val_dl), batch_size=128, shuffle=False,
                             collate_fn=collate_words)
test_word_loader = DataLoader(WordDataset(test_texts_dl, y_test_dl), batch_size=128, shuffle=False,
                              collate_fn=collate_words)

print(f'Vocabulary (train only): {len(vocab):,}')
print(f'Train/Val/Test: {len(y_train_dl):,}/{len(y_val_dl):,}/{len(y_test_dl):,}')

### 9.2 Shared PyTorch training and evaluation

Validation F1 ayaa lagu doortaa checkpoint-ka ugu fiican. Test set-ka lama adeegsanayo model selection.

In [ ]:
def binary_metrics(y_true_ids, y_pred_ids):
    return {
        'Accuracy': accuracy_score(y_true_ids, y_pred_ids),
        'Precision': precision_score(y_true_ids, y_pred_ids, average='weighted', zero_division=0),
        'Recall': recall_score(y_true_ids, y_pred_ids, average='weighted', zero_division=0),
        'F1': f1_score(y_true_ids, y_pred_ids, average='weighted', zero_division=0),
        'Crime Recall': recall_score(y_true_ids, y_pred_ids, pos_label=1, zero_division=0),
    }

@torch.no_grad()
def evaluate_word_model(model, loader):
    model.eval()
    truth, predictions = [], []
    for ids, lengths, labels in loader:
        logits = model(ids.to(DEVICE), lengths.to(DEVICE))
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        truth.extend(labels.tolist())
    return binary_metrics(truth, predictions), predictions

def train_word_model(model, name, epochs):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    best_state, best_f1, patience_left = None, -1.0, 2
    history = []
    started = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for ids, lengths, labels in train_word_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids.to(DEVICE), lengths.to(DEVICE))
            loss = criterion(logits, labels.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(labels)

        val_metrics, _ = evaluate_word_model(model, val_word_loader)
        epoch_loss = total_loss / len(train_word_loader.dataset)
        history.append({'Epoch': epoch, 'Loss': epoch_loss, 'Val F1': val_metrics['F1']})
        print(f'{name} | epoch {epoch:02d} | loss={epoch_loss:.4f} | val_f1={val_metrics["F1"]:.4f}')

        if val_metrics['F1'] > best_f1 + 1e-4:
            best_f1 = val_metrics['F1']
            best_state = copy.deepcopy(model.state_dict())
            patience_left = 2
        else:
            patience_left -= 1
            if patience_left == 0:
                print('Early stopping')
                break

    model.load_state_dict(best_state)
    test_metrics, predictions = evaluate_word_model(model, test_word_loader)
    test_metrics.update({'Model': name, 'Family': 'Deep Learning',
                         'Training seconds': time.time() - started,
                         'Mode': 'full same split' if not FAST_RUN else 'fast subset'})
    return model, test_metrics, pd.DataFrame(history), predictions

### 9.3 1D-CNN

Multiple kernel sizes waxay qabtaan weedho dembiyeed gaagaaban oo dhererkoodu kala duwan yahay.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, channels=96, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, channels, k, padding=k // 2)
                                    for k in (3, 5, 7)])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(channels * 3, 2)

    def forward(self, ids, lengths=None):
        x = self.embedding(ids).transpose(1, 2)
        features = [torch.relu(conv(x)).amax(dim=2) for conv in self.convs]
        return self.classifier(self.dropout(torch.cat(features, dim=1)))

cnn_search = []
for cfg in [
    {'embed_dim': 96, 'channels': 64, 'dropout': 0.30},
    {'embed_dim': 128, 'channels': 96, 'dropout': 0.40},
]:
    candidate = train_word_model(
        TextCNN(len(vocab), **cfg), '1D-CNN candidate', epochs=1 if FAST_RUN else 10
    )
    cnn_search.append((candidate[2]['Val F1'].max(), cfg, candidate))
cnn_best_val_f1, cnn_best_params, cnn_best = max(cnn_search, key=lambda item: item[0])
cnn_model, cnn_metrics, cnn_history, cnn_predictions = cnn_best
cnn_metrics['Model'] = '1D-CNN'
cnn_metrics['Best Params'] = str(cnn_best_params)
print('CNN best validation F1:', round(cnn_best_val_f1, 4), cnn_best_params)
display(pd.DataFrame([cnn_metrics]).set_index('Model').round(4))

### 9.4 BiLSTM

Packed sequences waxay ka ilaalinayaan LSTM inuu padding-ka u qaato qoraal dhab ah.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=96, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, ids, lengths):
        embedded = self.embedding(ids)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        representation = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(representation))

bilstm_search = []
for cfg in [
    {'embed_dim': 96, 'hidden_dim': 64, 'dropout': 0.30},
    {'embed_dim': 128, 'hidden_dim': 96, 'dropout': 0.40},
]:
    candidate = train_word_model(
        BiLSTMClassifier(len(vocab), **cfg), 'BiLSTM candidate', epochs=1 if FAST_RUN else 10
    )
    bilstm_search.append((candidate[2]['Val F1'].max(), cfg, candidate))
bilstm_best_val_f1, bilstm_best_params, bilstm_best = max(bilstm_search, key=lambda item: item[0])
bilstm_model, bilstm_metrics, bilstm_history, bilstm_predictions = bilstm_best
bilstm_metrics['Model'] = 'BiLSTM'
bilstm_metrics['Best Params'] = str(bilstm_best_params)
print('BiLSTM best validation F1:', round(bilstm_best_val_f1, 4), bilstm_best_params)
display(pd.DataFrame([bilstm_metrics]).set_index('Model').round(4))

### 9.5 Transformer preparation

Transformers-ku waxay helayaan **raw text**, sababtoo ah tokenizer-kooda ayaa u baahan qaabka ereyada oo aan stopwords laga saarin. FAST mode-ku wuxuu yareeyaa train/validation oo keliya; test set-ku weli waa isla hold-out test-ka oo dhan.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

raw_train = df.loc[X_train_text.index, 'text'].astype(str).tolist()
raw_val = df.loc[X_val_text.index, 'text'].astype(str).tolist()
raw_test = df.loc[X_test_text.index, 'text'].astype(str).tolist()

def stratified_limit(texts, labels, limit):
    if limit is None or len(labels) <= limit:
        return list(texts), list(labels)
    selected, _ = train_test_split(
        np.arange(len(labels)), train_size=limit, random_state=SEED, stratify=labels
    )
    return [texts[i] for i in selected], [int(labels[i]) for i in selected]

transformer_train_texts, transformer_y_train = stratified_limit(
    raw_train, y_train_all_dl, 100 if FAST_RUN else None
)
transformer_val_texts, transformer_y_val = stratified_limit(
    raw_val, y_val_all_dl, 40 if FAST_RUN else None
)
transformer_test_texts, transformer_y_test = stratified_limit(
    raw_test, y_test_all_dl, 200 if FAST_RUN else None
)
print('Transformer train/val/test:', len(transformer_y_train), len(transformer_y_val), len(transformer_y_test))

### 9.6 Shared transformer fine-tuning

Fine-tuning-ka wuxuu adeegsadaa AdamW, gradient clipping, validation model selection, iyo isla hold-out test-ka.

In [ ]:
class TransformerTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels = list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index], truncation=True, max_length=self.max_length,
            padding=False, return_tensors=None
        )
        encoded['labels'] = int(self.labels[index])
        return encoded

def make_transformer_loader(texts, labels, tokenizer, batch_size, shuffle, max_length):
    dataset = TransformerTextDataset(texts, labels, tokenizer, max_length)
    def collate(batch):
        labels = torch.tensor([item.pop('labels') for item in batch], dtype=torch.long)
        padded = tokenizer.pad(batch, padding=True, return_tensors='pt')
        padded['labels'] = labels
        return padded
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

@torch.no_grad()
def evaluate_transformer(model, loader):
    model.eval()
    truth, predictions = [], []
    for batch in loader:
        labels = batch.pop('labels')
        output = model(**{k: v.to(DEVICE) for k, v in batch.items()})
        predictions.extend(output.logits.argmax(dim=1).cpu().tolist())
        truth.extend(labels.tolist())
    return binary_metrics(truth, predictions), predictions

def train_transformer(checkpoint, display_name, family, max_length=128,
                      learning_rate=2e-5, full_epochs=3, weight_decay=0.01):
    print(f'Loading {checkpoint} ...')
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=2,
        id2label={0: NON_CRIME_LABEL, 1: CRIME_LABEL},
        label2id={NON_CRIME_LABEL: 0, CRIME_LABEL: 1},
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    batch_size = 8 if DEVICE.type == 'cuda' else 4
    train_loader = make_transformer_loader(transformer_train_texts, transformer_y_train,
                                           tokenizer, batch_size, True, max_length)
    val_loader = make_transformer_loader(transformer_val_texts, transformer_y_val,
                                         tokenizer, batch_size * 2, False, max_length)
    test_loader = make_transformer_loader(transformer_test_texts, transformer_y_test, tokenizer,
                                          batch_size * 2, False, max_length)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    epochs = 1 if FAST_RUN else full_epochs
    best_state, best_f1, history = None, -1.0, []
    started = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for step, batch in enumerate(train_loader, 1):
            optimizer.zero_grad(set_to_none=True)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            output = model(**batch)
            output.loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += output.loss.item()
            if step % 50 == 0:
                print(f'{display_name} epoch {epoch} step {step}/{len(train_loader)} loss={running/step:.4f}')

        val_metrics, _ = evaluate_transformer(model, val_loader)
        history.append({'Epoch': epoch, 'Loss': running / max(1, len(train_loader)),
                        'Val F1': val_metrics['F1']})
        print(f'{display_name} | epoch {epoch} | val_f1={val_metrics["F1"]:.4f}')
        if val_metrics['F1'] > best_f1:
            best_f1 = val_metrics['F1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_metrics, predictions = evaluate_transformer(model, test_loader)
    test_metrics.update({
        'Model': display_name, 'Family': family,
        'Training seconds': time.time() - started,
        'Mode': f'{"fast" if FAST_RUN else "full"}: train={len(transformer_y_train)}, epochs={epochs}'
    })
    return model, tokenizer, test_metrics, pd.DataFrame(history), predictions

### 9.7 SomBERTa-A — encoder-only

Checkpoint-ka waa SomBERTa rasmiga ah. Variant A wuxuu adeegsadaa sequence 128, learning rate 2e-5, iyo 3 epochs marka full training la sameeyo.

In [ ]:
SOMBERTA_CHECKPOINT = 'shuabdaud/SomBERTa'
somberta_a_model, somberta_a_tokenizer, somberta_a_metrics, somberta_a_history, somberta_a_predictions = train_transformer(
    SOMBERTA_CHECKPOINT, 'SomBERTa-A', 'Encoder-only Transformer', max_length=128,
    learning_rate=2e-5, full_epochs=3, weight_decay=0.01
)
display(pd.DataFrame([somberta_a_metrics]).set_index('Model').round(4))

### 9.8 SomBERTa-B — encoder-only

Variant B wuxuu adeegsadaa sequence 256, learning rate 1e-5, weight decay 0.02, iyo 4 epochs marka full training la sameeyo. Labada variant waxay isticmaalaan isla checkpoint-ka, laakiin fine-tuning configuration-koodu waa kala duwan yahay.

In [ ]:
# Release variant A before loading variant B to reduce RAM pressure.
del somberta_a_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

somberta_b_model, somberta_b_tokenizer, somberta_b_metrics, somberta_b_history, somberta_b_predictions = train_transformer(
    SOMBERTA_CHECKPOINT, 'SomBERTa-B', 'Encoder-only Transformer', max_length=256,
    learning_rate=1e-5, full_epochs=4, weight_decay=0.02
)
display(pd.DataFrame([somberta_b_metrics]).set_index('Model').round(4))

## 10. Final model comparison

Jadwalkani wuxuu isu keenayaa classical ML, deep learning, iyo transformers. FAST transformer rows waa preliminary; final claim samee kaddib `FAST_RUN=False` run.

In [ ]:
# Normalize the existing classical hold-out results to the same schema.
classical_results = test_df.copy()
classical_results['Family'] = 'Classical ML'
classical_results['Training seconds'] = np.nan
classical_results['Mode'] = 'full split'
classical_results['Precision'] = classical_results['Crime Precision']
classical_results['Recall'] = classical_results['Crime Recall']

neural_rows = pd.DataFrame([cnn_metrics, bilstm_metrics, somberta_a_metrics, somberta_b_metrics])
comparison_columns = ['Model', 'Family', 'Accuracy', 'Precision', 'Recall', 'F1',
                      'Crime Recall', 'Training seconds', 'Mode']
all_model_results = pd.concat(
    [classical_results[comparison_columns], neural_rows[comparison_columns]],
    ignore_index=True
).sort_values(['Crime Recall', 'F1'], ascending=False).reset_index(drop=True)

display(all_model_results.style.format({
    'Accuracy': '{:.3f}', 'Precision': '{:.3f}', 'Recall': '{:.3f}',
    'F1': '{:.3f}', 'Crime Recall': '{:.3f}', 'Training seconds': '{:.1f}'
}).background_gradient(subset=['F1', 'Crime Recall'], cmap='YlGn'))

results_path = OUTPUT_DIR.parent / 'all_model_results.csv'
all_model_results.to_csv(results_path, index=False)
print(f'Saved: {results_path.resolve()}')

In [ ]:
plot_results = all_model_results.sort_values('F1')
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(plot_results['Model'], plot_results['F1'], color='#2e86de', edgecolor='black')
axes[0].set_title('Hold-out Test F1 by Model')
axes[0].set_xlim(0, 1.03)
axes[0].set_xlabel('F1')

axes[1].barh(plot_results['Model'], plot_results['Crime Recall'], color='#c0392b', edgecolor='black')
axes[1].set_title('Hold-out Test Crime Recall by Model')
axes[1].set_xlim(0, 1.03)
axes[1].set_xlabel('Crime Recall')

for ax in axes:
    ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
try:
    plt.savefig(OUTPUT_DIR / '09_all_model_comparison.png', dpi=180, bbox_inches='tight')
except OSError as chart_error:
    print(f'Chart save skipped: {chart_error}')
plt.show()

In [ ]:
# Select by the shared weighted-F1 metric, then crime recall.
best_row = all_model_results.sort_values(['F1', 'Crime Recall'], ascending=False).iloc[0]
print('BEST CURRENT MODEL (F1, then Crime Recall)')
print('-' * 48)
print(f'Model:        {best_row["Model"]}')
print(f'Family:       {best_row["Family"]}')
print(f'F1:           {best_row["F1"]:.4f}')
print(f'Crime Recall: {best_row["Crime Recall"]:.4f}')
print(f'Mode:         {best_row["Mode"]}')
assert not FAST_RUN, 'Do not publish subset metrics as final results.'

## 11. Save Production Artifacts


In [ ]:
if not hasattr(production_model, 'predict_proba'):
    print('Wrapping with CalibratedClassifierCV...')
    calibrator = CalibratedClassifierCV(production_model, method='sigmoid', cv=3)
    calibrator.fit(X_tune, y_tune)
    production_model = calibrator

model_path = AI_MODEL_DIR / 'crime_model.pkl'
vectorizer_path = AI_MODEL_DIR / 'vectorizer.pkl'
joblib.dump(production_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

meta = {
    'model_name': production_name,
    'feature_config': best_feature_name,
    'n_features': int(X_train.shape[1]),
    'train_size': int(X_train.shape[0]),
    'test_f1': float(f1_score(y_test, production_model.predict(X_test), average='weighted')),
    'test_crime_recall': float(recall_score(y_test, production_model.predict(X_test), pos_label=CRIME_LABEL)),
    'preprocess': 'ai-model/preprocessing.py::preprocess_text',
    'stopwords_source': 'model sax + extras (somali_stopwords.json)',
}
joblib.dump(meta, AI_MODEL_DIR / 'model_meta.pkl')
print('Saved:', model_path)
print('Saved:', vectorizer_path)
print('Meta:', meta)
print('Restart AI API (port 5001) after save.')


## 12. Smoke test (API path)


In [ ]:
samples = [
    'Nin ayaa lagu dilay magaalada Muqdisho ee degmada Hodan habeen hore.',
    'Ciyaaraha football-ka ayaa caawa ka dhacaya garoonka Muqdisho Stadium.',
    'Qarax ayaa ka dhacay suuqa, dad badan ayaa ku dhaawacmay.',
]
loaded_model = joblib.load(AI_MODEL_DIR / 'crime_model.pkl')
loaded_vec = joblib.load(AI_MODEL_DIR / 'vectorizer.pkl')
print('Smoke test:')
for s in samples:
    processed = preprocess_text(s)
    vec = loaded_vec.transform([processed])
    pred = loaded_model.predict(vec)[0]
    conf = max(loaded_model.predict_proba(vec)[0]) * 100 if hasattr(loaded_model, 'predict_proba') else float('nan')
    print(f'  [{pred}] ({conf:.1f}%) :: {s[:70]}')


In [ ]:
# Download all final thesis outputs as one ZIP.
import shutil
archive = shutil.make_archive('/content/BareAI_Final_Thesis_Results', 'zip', AI_MODEL_DIR)
print('Results archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('Download the ZIP manually from the Colab Files panel.')